# 01 — Data Collection

## UAE Used Car Market Analysis

This project pulls used car listings from three UAE marketplaces —
CarSwitch, AutoTraders.ae, and OpenSooq — and combines them into a single
dataset for analysis. This notebook covers where the data came from, how
each site was scraped, and a few sources that were tried and dropped along
the way.

A live scrape across all three sources takes somewhere between one and
two hours, so this notebook doesn't re-run the actual scraping — that
logic lives in `src/` as standalone scripts (`extractors/autotraders.py`,
`extractors/opensooq.py`, `dedup.py`, and the CarSwitch scraper). What's
here is the methodology plus the real output loaded from the saved CSVs.

## Sources

| Source | Status | Approx. listings |
|---|---|---|
| CarSwitch | Scraped | 2,357 |
| AutoTraders.ae | Scraped | ~17,500 |
| OpenSooq | Scraped | ~9,200 |
| Dubizzle | Not used | Blocked by an Imperva WAF — wouldn't budge for a plain request or Selenium |
| YallaMotor | Not used | Cloudflare bot challenge on every listing page |
| DubiCars | Not used | Open at first, then started throwing a Cloudflare challenge partway through the project |
| Hatla2ee | Not used | robots.txt disallows the search path, and the UAE inventory was tiny anyway (~400 listings) |
| Kavak | Not pursued | Listings load through a client-side API call rather than server-rendered HTML — would've needed the specific endpoint, and the other three sources already gave enough volume |

The rule followed throughout: if a site puts up active bot detection, it
stays off the list. No browser automation, no fingerprint spoofing, no
CAPTCHA workarounds — just plain requests to whatever would actually
answer them.

## CarSwitch

`carswitch.com/uae/used-cars/search`

CarSwitch embeds its listings as JSON-LD structured data right in the
page — no HTML parsing needed, the fields come straight out of the
`<script type="application/ld+json">` block.

The one wrinkle was rate limiting: after roughly 18-20 pages in a session,
the site starts returning HTTP 202s instead of real pages. The workaround
was a batch-and-resume pattern — 18 pages per batch, a fresh session each
time, a 10-minute pause between batches, and random 5-9 second gaps
between individual page requests.

106 pages later (page 107 came back empty, confirming the end of
inventory), deduplicating by URL landed on 2,357 unique listings — the
dedup step matters here because it's a live marketplace and listings
shift position while you're still working through the batches.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")
carswitch = pd.read_csv(data_dir / "raw" / "carswitch_master_raw.csv")
print(f"{len(carswitch)} listings")
carswitch.head()

2357 listings


,name,brand,model,year,transmission,mileage_km,price_aed,city,url
0,2020 Dodge RAM 1500 Limited,Dodge,RAM,2020,Automatic,198250,79500,abudhabi,https://carswitch.com/abudhabi/used-car/dodge/...
1,2022 MINI Cooper Twin Power Turbo,MINI,Cooper,2022,Automatic,50350,56000,dubai,https://carswitch.com/dubai/used-car/mini/coop...
2,2017 Infiniti QX70,Infiniti,QX70,2017,Automatic,76000,35000,abudhabi,https://carswitch.com/abudhabi/used-car/infini...
3,2018 Jeep Grand Cherokee Limited,Jeep,Grand Cherokee,2018,Automatic,110600,40000,dubai,https://carswitch.com/dubai/used-car/jeep/gran...
4,2024 Tesla Model Y Long Range Dual Motor AWD,Tesla,Model Y,2024,Automatic,69500,120900,abudhabi,https://carswitch.com/abudhabi/used-car/tesla/...


## AutoTraders.ae

`uae.autotraders.ae/used-cars`

Server-rendered HTML, each listing sitting inside a `div.car-card`:

- Title lives in `h2.car-card-title`
- Brand and model come from `div.car-card-subline`, formatted as
  `"Brand  -  Model"` — worth being careful splitting on that hyphen,
  since names like "Mercedes-Benz" or "G-Class" have their own internal
  hyphens that shouldn't get caught by the split
- Price sits in `div.car-card-price-amount`
- Year and mileage show up as little chip elements (`"Year 2025"`,
  `"KM 8,700"`)
- The detail link's URL slug conveniently spells out brand, model, trim,
  and year on its own (`/used-cars/kia/k5/no-trim/2025/...`), which turned
  out more reliable than parsing it back out of the card text

Pagination is a plain `?page=N`, confirmed against the site's own "next"
control — about 730 pages total.

One thing that ate more time than it should have: the first several
attempts at this page came back completely empty, no listings at all,
which looked at first like either a bot wall or a JavaScript-only app.
Turned out to be a self-inflicted bug — an `Accept-Encoding: br` header
was being sent without Brotli decompression support installed locally, so
the response body was silently corrupted into unreadable bytes before it
ever reached the parser. Dropping Brotli from the request headers fixed
it instantly and revealed the site had been fully scrapable server-rendered
HTML the whole time.

About 22.6% of AutoTraders listings don't show a price at all — these
read as "Call for Price" dealer listings, which lines up with the site
skewing toward dealer and luxury inventory generally. A smaller slice
(under half a percent) had garbled price or mileage values traced to a
handful of listing cards using a different internal template than the
standard one — those get flagged rather than trusted, in the cleaning
notebook.

In [2]:
autotraders = pd.read_csv(data_dir / "raw" / "autotraders_master_raw.csv")
print(f"{len(autotraders)} listings")
print(f"Missing price: {autotraders['price_aed'].isna().sum()} ({100*autotraders['price_aed'].isna().mean():.1f}%)")
autotraders.head()

17505 listings
Missing price: 4067 (23.2%)


,source,listing_id,name,brand,model,year,transmission,mileage_km,price_aed,city,url
0,autotraders,d6afb9124667,KIA SPORTAGE SX PRESTIGE 2025,Kia,Sportage,2025,NaN,9300.0,95000.0,Ajman,https://uae.autotraders.ae/used-cars/kia/sport...
1,autotraders,5c3c9bea01b5,Kia K5 GT LINE 2025,Kia,K5,2025,NaN,8700.0,95000.0,Ajman,https://uae.autotraders.ae/used-cars/kia/k5/no...
2,autotraders,269e1ff2cb69,2025 MERCEDES BENZ AMG G63 / DOUBLE NIGHT PACK...,Mercedes-Benz,G-Class,2025,NaN,9700.0,999000.0,Dubai,https://uae.autotraders.ae/used-cars/mercedes-...
3,autotraders,65b2820d81b2,2023 RANGE ROVER SV LWB / WARRANTY AL TAYER / ...,Land Rover,Range Rover,2023,NaN,22600.0,680000.0,Dubai,https://uae.autotraders.ae/used-cars/land-rove...
4,autotraders,1ee2d0de0cb8,2024 Rolls Royce Spectre,Rolls Royce,SPECTRE,2024,NaN,6600.0,1300000.0,Dubai,https://uae.autotraders.ae/used-cars/rolls-roy...


## OpenSooq

`ae.opensooq.com/en/cars/cars-for-sale`

OpenSooq runs on Next.js, and the listings aren't sitting in plain HTML
or JSON-LD — they're buried inside a `<script id="__NEXT_DATA__">` blob.
Finding the actual path took a bit of trial and error before settling on
just dumping the raw JSON and reading it directly instead of guessing:

```
data["props"]["pageProps"]["serpApiResponse"]["listings"]["items"]
```

Getting there wasn't completely smooth — an earlier approach that just
recursively hunted for "the biggest array of dicts" in the JSON blob
grabbed the wrong thing twice: first a brand/model filter list (tiny
internal category IDs, no price data), then a financing/installment
options array where the same car showed up several times with a
different price for each payment plan. Once the actual JSON structure was
inspected directly rather than guessed at, the real path took about a
minute to find.

Pagination is `?page=N`, confirmed straight from the page's own links.

Final count sits around 9,000-11,000 depending on exactly when it's
pulled — it's a live marketplace, so the number moves as listings get
posted and taken down.

In [3]:
opensooq = pd.read_csv(data_dir / "raw" / "opensooq_master_raw.csv")
print(f"{len(opensooq)} listings")
opensooq.head()

9177 listings


,source,listing_id,name,brand,model,year,transmission,mileage_km,price_aed,city,url
0,opensooq,5c7f36648dff,2022 Nissan Patrol Safari,Nissan,Patrol,2022,NaN,92000.0,178000.0,Abu Dhabi,https://ae.opensooq.com/search/282243718
1,opensooq,6c65a10c441a,2020 Lexus UX UX 300h,Lexus,UX,2020,NaN,122000.0,62000.0,Um Al Quwain,https://ae.opensooq.com/search/283292712
2,opensooq,f6050d3daf5a,2022 Nissan Frontier Crew Cab SV,Nissan,Frontier,2022,NaN,61300.0,60000.0,Um Al Quwain,https://ae.opensooq.com/search/279819721
3,opensooq,2ec5b0f15cf9,2022 Ford Territory,Ford,Territory,2022,NaN,80000.0,47000.0,Ajman,https://ae.opensooq.com/search/282827174
4,opensooq,832160a93bfa,2020 Kia Forte GT-Line,Kia,Forte,2020,NaN,150000.0,22000.0,Ajman,https://ae.opensooq.com/search/282970582


## Combining the three

Dealers commonly post the same car on more than one platform, so before
combining everything, a fuzzy-match dedup step runs across all three
sources: a hash of `brand + model + year + price (rounded to the nearest
1,000 AED) + mileage (rounded to the nearest 1,000 km)`. Loose enough that
the same car listed with slightly different rounding on two sites still
matches, tight enough not to accidentally merge two genuinely different
cars.

This turned out to be worth measuring in its own right, not just a
cleanup step — it tells you how much of the combined listing count is
actually duplicate inventory versus real distinct supply.

In [4]:
import sys
sys.path.append("../scraper")
from dedup import merge_and_dedup

merge_and_dedup(
    data_dir=data_dir,
    output_path=data_dir / "processed" / "uae_cars_combined_deduped.csv",
)

2026-07-03 04:26:34,058 | INFO     | Loaded 2357 rows from carswitch
2026-07-03 04:26:34,256 | INFO     | Loaded 17505 rows from autotraders
2026-07-03 04:26:34,350 | INFO     | Loaded 9177 rows from opensooq



--- Cross-source overlap ---
carswitch <-> autotraders: 4 shared listings (0.2% of carswitch, 0.0% of autotraders)
carswitch <-> opensooq: 3 shared listings (0.1% of carswitch, 0.0% of opensooq)
autotraders <-> opensooq: 52 shared listings (0.3% of autotraders, 0.6% of opensooq)

Combined rows before dedup: 29039
Combined rows after dedup:  28109
Removed as duplicates:      930 (3.2%)
Saved -> ..\data\processed\uae_cars_combined_deduped.csv


Overlap turned out smaller than expected:

| Pair | Shared listings |
|---|---|
| CarSwitch ↔ AutoTraders.ae | 4 |
| CarSwitch ↔ OpenSooq | 3 |
| AutoTraders.ae ↔ OpenSooq | 52 |

930 duplicates removed out of 29,039 combined raw rows — about 3.2% —
leaving 28,109 unique listings. Given how much dealer cross-posting
happens in this market generally, this is lower than I'd have guessed
going in, which suggests AutoTraders.ae and OpenSooq are pulling from
largely different pools of actual inventory rather than the same cars
showing up twice under different usernames.

## Where things land

| Source | Listings |
|---|---|
| CarSwitch | 2,357 |
| AutoTraders.ae | ~17,505 |
| OpenSooq | ~9,177 |
| Combined, before dedup | 29,039 |
| Combined, after dedup | 28,109 |

Saved to `data/processed/uae_cars_combined_deduped.csv`.

Next up: `02_cleaning.ipynb` — brand and model name normalization, city
standardization, and the feature engineering that turns this into
something ready for analysis.